In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

In [ ]:
data_train = pd.read_csv('period_1_train_data.csv')
data_test = pd.read_csv('test_x.csv')
test_ids = data_test['id'].copy()

print(data_train.head(10))
print(f"NaN в train: {data_train.isna().sum().sum()}")
print(f"NaN в test: {data_test.isna().sum().sum()}")

data_train = data_train.fillna(-999)
data_test = data_test.fillna(-999)

  region_name_cat  district_cat  corpus_cat  developer_cat agreement_date  \
0           Город            45         538             18     2012-08-10   
1        Пригород            48         432             63     2013-05-19   
2           Город            44        2372            126     2012-12-12   
3           Город            14        1053            121     2012-12-10   
4           Город            63        2426             69     2012-02-12   
5           Город            59         756             91     2013-01-30   
6           Город             7         970             86     2012-06-05   
7           Город            53        2088             97     2012-03-31   
8           Город             7         589             86     2012-04-12   
9           Город            63        1446             69     2013-04-10   

   floor  square rooms_4  location_logs_count_mean  location_depth  ...  \
0    3.0   62.23       2                 22.550466            13.0  ...   
1 

In [ ]:
data_train['agreement_date'] = pd.to_datetime(data_train['agreement_date'])
data_test['agreement_date'] = pd.to_datetime(data_test['agreement_date'])

for dataset in [data_train, data_test]:
    dataset['agree_year'] = dataset['agreement_date'].dt.year
    dataset['agree_month'] = dataset['agreement_date'].dt.month
    dataset['agree_weekday'] = dataset['agreement_date'].dt.dayofweek
    dataset['agree_daynum'] = dataset['agreement_date'].dt.dayofyear
    dataset.drop('agreement_date', axis=1, inplace=True)

In [9]:
room_conversion = {'студия': 0, '1': 1, '2': 2, '3': 3, '>=4': 4}
data_train['rooms_count'] = data_train['rooms_4'].map(room_conversion).fillna(0)
data_test['rooms_count'] = data_test['rooms_4'].map(room_conversion).fillna(0)

data_train.drop('rooms_4', axis=1, inplace=True)
data_test.drop('rooms_4', axis=1, inplace=True)

In [ ]:
city_mapping = {'Город': 1, 'Пригород': 0}
data_train['is_city'] = data_train['region_name_cat'].map(city_mapping).fillna(0)
data_test['is_city'] = data_test['region_name_cat'].map(city_mapping).fillna(0)


data_train.drop('region_name_cat', axis=1, inplace=True)
data_test.drop('region_name_cat', axis=1, inplace=True)

In [11]:
target_col = 'price_target'
X = data_train.drop(target_col, axis=1)
y = data_train[target_col]

In [12]:
X_full = X.copy()

for col in X_full.columns:
    if col not in data_test.columns:
        data_test[col] = 0

for col in data_test.columns:
    if col not in X_full.columns:
        data_test = data_test.drop(col, axis=1)
        
data_test = data_test[X_full.columns]

In [13]:
model_params = {
    'n_estimators': 1000,
    'max_depth': 30,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'n_jobs': -1
}

random_seeds = [42, 123, 456, 789, 999]
ensemble_models = []
all_forecasts = []


for idx, seed in enumerate(random_seeds, 1):
    print(f"\n[Модель {idx}/5] random_state={seed}")
    
    current_model = ExtraTreesRegressor(**model_params, random_state=seed)
    current_model.fit(X_full, y)
    ensemble_models.append(current_model)
    
    X_tr, X_val, y_tr, y_val = train_test_split(X_full, y, test_size=0.2, random_state=seed)
    temp_model = ExtraTreesRegressor(**model_params, random_state=seed)
    temp_model.fit(X_tr, y_tr)
    temp_pred = temp_model.predict(X_val)
    temp_mape = mean_absolute_percentage_error(y_val, temp_pred)
    print(f"    MAPE на валидации: {temp_mape:.4f} ({temp_mape*100:.2f}%)")
    
    forecast = current_model.predict(data_test)
    all_forecasts.append(forecast)



[Модель 1/5] random_state=42
    MAPE на валидации: 0.0178 (1.78%)

[Модель 2/5] random_state=123
    MAPE на валидации: 0.0185 (1.85%)

[Модель 3/5] random_state=456
    MAPE на валидации: 0.0180 (1.80%)

[Модель 4/5] random_state=789
    MAPE на валидации: 0.0181 (1.81%)

[Модель 5/5] random_state=999
    MAPE на валидации: 0.0181 (1.81%)


In [ ]:
final_forecast = np.mean(all_forecasts, axis=0)

price_lower = y.quantile(0.005)
price_upper = y.quantile(0.995)
final_forecast = np.clip(final_forecast, price_lower, price_upper)

print(f"Минимальное: {final_forecast.min():.2f}")
print(f"Максимальное: {final_forecast.max():.2f}")
print(f"Среднее: {final_forecast.mean():.2f}")
print(f"Медиана: {np.median(final_forecast):.2f}")

Минимальное: 15909.81
Максимальное: 85444.24
Среднее: 24494.73
Медиана: 22353.29


In [ ]:
submission_file = pd.DataFrame({
    'id': test_ids,
    'price_target': final_forecast
})
submission_file.to_csv('submission.csv', index=False)

submission_file.head(10)

,id,price_target
0,0,22176.860726
1,1,23367.092909
2,2,27487.678169
3,3,18801.918161
4,4,19672.312268
5,5,18099.122837
6,6,21577.256791
7,7,59875.596172
8,8,24384.136395
9,9,16545.175606
